# Regression NBA Model


## Configuration

## Imports

In [48]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training,
)
from nba_ou.modeling.modeling import (
    TemporalDecaySampleWeightRegressor,
    evaluate_day_by_day_walk_forward,
    split_latest_dates_holdout,
    make_walk_forward_last_n_seasons_splits,
    validate_time_splits,
    make_test_anchored_walk_forward_splits,
    assert_valid_time_splits,
    save_model_bundle,
    load_model_bundle,
)


In [49]:
TARGET_COL = "LINE_ERROR"
SAMPLE_WEIGHT_LAMBDA = 0.005
SAMPLE_WEIGHT_LAMBDA_BOUNDS = (1e-4, 0.01)
TRAIN_GAMES = 5000

## Load Data

In [50]:
nan_threshold = 50.0
max_na_per_row = 80


data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260405.csv"

path = data_path + name

header_cols = pd.read_csv(path, nrows=0).columns
dtype_dict = {col: str for col in header_cols if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict,
)
df_stats["GAME_DATE"] = pd.to_datetime(df_stats["GAME_DATE"]).dt.strftime("%Y-%m-%d")
df_stats = df_stats[df_stats['SEASON_YEAR'] >= 2021].copy()


In [51]:
exclude = "fanatics_sportsbook"

In [52]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'], exclude_cols_containing=[exclude])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 6445 rows
Basic cleaning complete: 6430 rows remaining

Starting advanced column cleaning with 2948 columns

Advanced column cleaning complete: 2948 → 2054 columns (894 removed)


Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 112
  Infer pairs applied: 0/106
  Remaining NaN cells: 251290

Dropping rows with more than 80 NaN values...
Removed 618 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (5812, 2054)


In [53]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and "SEASON_YEAR" in df_to_train.columns:
            common_season = null_rows["SEASON_YEAR"].mode()
            most_common_season.append(
                common_season.iloc[0] if len(common_season) > 0 else None
            )
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame(
    {
        "Column": na_counts.index,
        "NA_Count": na_counts.values,
        "NA_Percentage": (na_counts.values / len(df_to_train) * 100).round(2),
        "Most_Common_Season_Year": most_common_season,
    }
).sort_values("NA_Count", ascending=False)

na_counts_df[na_counts_df["NA_Count"] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1732,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,792,13.63,2023.0
1730,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,778,13.39,2023.0
1736,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,709,12.20,2023.0
1734,spread_consensus_pct_away_TREND_SLOPE_LAST_5_H...,702,12.08,2023.0
1731,total_consensus_pct_under_TREND_SLOPE_LAST_5_G...,684,11.77,2023.0
...,...,...,...,...
1885,spread_betmgm_price_away,1,0.02,2021.0
1886,spread_betmgm_price_home,1,0.02,2021.0
1875,total_betmgm_price_under,1,0.02,2021.0
1874,total_betmgm_price_over,1,0.02,2021.0


In [54]:
BET365_LINE_COL = "TOTAL_LINE_bet365"
# BET365_LINE_COL = "total_bet365_line_over"

# Ensure the main scoring line and actual total exist.
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [55]:
df_to_train["LINE_ERROR"] = df_to_train["TOTAL_POINTS"] - df_to_train[BET365_LINE_COL]


In [56]:
df_to_train["GAME_DATE"] = pd.to_datetime(df_to_train["GAME_DATE"])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

# Count games per season
games_per_season = df_to_train.groupby("SEASON_YEAR").size()
print(games_per_season)


SEASON_YEAR
2021    1239
2022    1235
2023    1010
2024    1238
2025    1090
dtype: int64


## Train / Test

In [57]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
)
from sklearn.model_selection import cross_validate
from xgboost import XGBRegressor

from nba_ou.modeling.optuna_error_line import (
    fit_best_xgb_error_line,
    select_best_trial_lexicographic,
    summarize_lexicographic_candidates,
    summarize_optuna_trials,
    tune_xgb_error_line_optuna,
)
from nba_ou.modeling.scorers import (
    OverUnderScorerLineError,
    OverUnderScorerLineErrorMinEdge,
    evaluate_error_thresholds,
    over_under_betting_accuracy_error_line,
    over_under_betting_accuracy_error_line_with_min_edge,
)

In [58]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.08,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print(
    "Final test date range:",
    df_test_final["GAME_DATE"].min(),
    "->",
    df_test_final["GAME_DATE"].max(),
)

Development set size: 5341
Final test set size: 471
Final test date range: 2026-01-27 00:00:00 -> 2026-04-04 00:00:00


In [59]:
def build_recency_sample_weights(df, date_col="GAME_DATE", lambda_=SAMPLE_WEIGHT_LAMBDA):
    dates = pd.to_datetime(df[date_col])
    max_date = dates.max()
    age_days = (max_date - dates).dt.days
    weights = np.exp(-lambda_ * age_days)
    return pd.Series(weights, index=df.index, name="sample_weight")

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "LINE_ERROR",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")
sample_weight_dev = build_recency_sample_weights(df_dev)

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")
print(
    f"Recency sample weights lambda={SAMPLE_WEIGHT_LAMBDA}: "
    f"min={sample_weight_dev.min():.4f}, max={sample_weight_dev.max():.4f}"
)


X_dev shape: (5341, 2051)
X_test_final shape: (471, 2051)
Recency sample weights lambda=0.005: min=0.0004, max=1.0000


In [60]:
ou_scorer = OverUnderScorerLineError()
ou_scorer_edge_2 = OverUnderScorerLineErrorMinEdge(min_edge=2)
ou_scorer_edge_4 = OverUnderScorerLineErrorMinEdge(min_edge=4)

scoring = {
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
    "OU_Betting_Accuracy_Edge_2": ou_scorer_edge_2,
    "OU_Betting_Accuracy_Edge_4": ou_scorer_edge_4,
}


def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_val = cv_results[train_key].mean()
        test_val = cv_results[test_key].mean()

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if sc.startswith("OU_Betting_Accuracy"):
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()


In [61]:
DAY_BY_DAY_METRIC_NAME = "OU_Betting_Accuracy"
DAY_BY_DAY_THRESHOLDS = (1, 2, 3)


def summarize_walk_forward_thresholds(predictions_df, thresholds=DAY_BY_DAY_THRESHOLDS):
    y_true = pd.to_numeric(predictions_df["y_true"], errors="coerce").to_numpy(dtype=float)
    y_pred = pd.to_numeric(predictions_df["y_pred"], errors="coerce").to_numpy(dtype=float)
    margin = np.abs(y_pred)
    n_total = len(predictions_df)

    rows = []
    for t in thresholds:
        mask = margin > t
        n = int(mask.sum())
        acc = (
            np.nan
            if n == 0
            else over_under_betting_accuracy_error_line(
                y_true_error=y_true[mask],
                y_pred_error=y_pred[mask],
            )
        )
        rows.append(
            {
                "threshold_abs_pred_error_gt": t,
                "n_games": n,
                "pct_of_test": (n / n_total) if n_total else np.nan,
                "directional_accuracy": acc,
            }
        )

    return pd.DataFrame(rows)


def run_day_by_day_walk_forward_evaluation(
    *,
    label,
    df_dev,
    df_test_final,
    fit_and_predict,
    max_games=TRAIN_GAMES,
    metric_name=DAY_BY_DAY_METRIC_NAME,
    thresholds=DAY_BY_DAY_THRESHOLDS,
):
    result = evaluate_day_by_day_walk_forward(
        df_dev=df_dev,
        df_test_final=df_test_final,
        fit_and_predict=fit_and_predict,
        metric_fn=lambda y_true, y_pred: over_under_betting_accuracy_error_line(
            y_true_error=y_true,
            y_pred_error=y_pred,
        ),
        target_col=TARGET_COL,
        max_games=max_games,
        metric_name=metric_name,
    )

    threshold_results = summarize_walk_forward_thresholds(
        result.predictions,
        thresholds=thresholds,
    )

    print(f"{label} mean day-by-day {metric_name}: {result.mean_metric:.2%}")
    display(result.daily_results.style.format({metric_name: "{:.2%}"}))
    print(f"{label} thresholded walk-forward accuracy")
    display(
        threshold_results.style.format(
            {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
        )
    )
    return result, threshold_results


In [62]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=25,
    step_games_between_tests=50,
    train_games=TRAIN_GAMES,
    min_train_games=TRAIN_GAMES*0.75,
    max_folds=15,
    verbose=1,
)

assert_valid_time_splits(df_dev, splits)



Created 15 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           4085            28       2021-10-25     2025-01-25      2025-01-26    2025-01-29         2024
    2           4167            27       2021-10-25     2025-02-05      2025-02-06    2025-02-09         2024
    3           4247            30       2021-10-25     2025-02-21      2025-02-22    2025-02-25         2024
    4           4332            30       2021-10-25     2025-03-04      2025-03-05    2025-03-08         2024
    5           4418            33       2021-10-25     2025-03-15      2025-03-16    2025-03-19         2024
    6           4501            30       2021-10-25     2025-03-26      2025-03-27    2025-03-30         2024
    7           4587            27       2021-10-25     2025-04-06      2025-04-07    2025-04-10         2024
    8           4722            32       2021-10-25     2025-06-22      2025

In [63]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)


DummyRegressor baseline
Train MSE: 288.45885
Validation MSE: 308.74456

Train RMSE: 16.98405
Validation RMSE: 17.44733

Train MAE: 13.51229
Validation MAE: 13.78484

Train R2: 0.00000
Validation R2: -0.04183

Train OU_Betting_Accuracy: 51.64%
Validation OU_Betting_Accuracy: 52.54%

Train OU_Betting_Accuracy_Edge_2: 0.00%
Validation OU_Betting_Accuracy_Edge_2: 0.00%

Train OU_Betting_Accuracy_Edge_4: 0.00%
Validation OU_Betting_Accuracy_Edge_4: 0.00%



In [64]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)


Linear Regression
Train MSE: 187.87288
Validation MSE: 1165464.10105

Train RMSE: 13.70487
Validation RMSE: 300.91618

Train MAE: 10.78246
Validation MAE: 71.69889

Train R2: 0.34864
Validation R2: -2642.20472

Train OU_Betting_Accuracy: 69.94%
Validation OU_Betting_Accuracy: 49.50%

Train OU_Betting_Accuracy_Edge_2: 73.50%
Validation OU_Betting_Accuracy_Edge_2: 50.59%

Train OU_Betting_Accuracy_Edge_4: 77.14%
Validation OU_Betting_Accuracy_Edge_4: 50.27%



In [65]:
xgb_reg_no_weights = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=35,
    subsample=0.65,
    colsample_bytree=0.68,
    reg_alpha=5.28,
    reg_lambda=1.3,
    min_child_weight=5.08,
    gamma=0.0085,
    n_jobs=-1,
    random_state=16,
)

cv_results_no_weights = cross_validate(
    xgb_reg_no_weights,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost no sample weights")
print_metrics(cv_results_no_weights)

XGBoost no sample weights
Train MSE: 266.24405
Validation MSE: 307.27788

Train RMSE: 16.31692
Validation RMSE: 17.39952

Train MAE: 12.97005
Validation MAE: 13.70861

Train R2: 0.07699
Validation R2: -0.03607

Train OU_Betting_Accuracy: 63.22%
Validation OU_Betting_Accuracy: 54.38%

Train OU_Betting_Accuracy_Edge_2: 86.50%
Validation OU_Betting_Accuracy_Edge_2: 60.86%

Train OU_Betting_Accuracy_Edge_4: 98.46%
Validation OU_Betting_Accuracy_Edge_4: 6.67%



In [66]:
xgb_reg_no_weights.fit(X_dev, y_dev)

y_pred_test_error = xgb_reg_no_weights.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 305.24811
RMSE: 17.47135
MAE: 13.74194
OU_Betting_Accuracy: 55.75%
OU_Betting_Accuracy_Edge_2: 68.18%
OU_Betting_Accuracy_Edge_4: 100.00%


In [67]:
results_df, y_pred_test_error = evaluate_error_thresholds(
    model=xgb_reg_no_weights,
    X_test=X_test_final,
    y_test_error=y_test_final,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,471,100.0%,55.75%
1,1,164,34.8%,56.60%
2,2,24,5.1%,68.18%
3,3,1,0.2%,100.00%
4,4,1,0.2%,100.00%
5,5,0,0.0%,nan%
6,6,0,0.0%,nan%
7,7,0,0.0%,nan%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


In [68]:
def fit_and_predict_xgb_no_weights_day_by_day(train_df, test_df):
    model = XGBRegressor(**xgb_reg_no_weights.get_params())

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_no_weights, day_by_day_no_weights_thresholds = run_day_by_day_walk_forward_evaluation(
    label="XGBoost no sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_no_weights_day_by_day,
    max_games=TRAIN_GAMES,
)

XGBoost no sample weights mean day-by-day OU_Betting_Accuracy: 55.02%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-01-27 00:00:00,5000,7,2021-12-14 00:00:00,2026-01-26 00:00:00,66.67%
1,2026-01-28 00:00:00,5000,9,2021-12-15 00:00:00,2026-01-27 00:00:00,88.89%
2,2026-01-29 00:00:00,5000,8,2021-12-16 00:00:00,2026-01-28 00:00:00,75.00%
3,2026-01-30 00:00:00,5000,9,2021-12-17 00:00:00,2026-01-29 00:00:00,37.50%
4,2026-01-31 00:00:00,5000,6,2021-12-19 00:00:00,2026-01-30 00:00:00,50.00%
5,2026-02-01 00:00:00,5000,10,2021-12-20 00:00:00,2026-01-31 00:00:00,70.00%
6,2026-02-02 00:00:00,5000,4,2021-12-22 00:00:00,2026-02-01 00:00:00,50.00%
7,2026-02-03 00:00:00,5000,10,2021-12-22 00:00:00,2026-02-02 00:00:00,30.00%
8,2026-02-04 00:00:00,5000,7,2021-12-23 00:00:00,2026-02-03 00:00:00,42.86%
9,2026-02-05 00:00:00,5000,8,2021-12-26 00:00:00,2026-02-04 00:00:00,50.00%


XGBoost no sample weights thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,200,42.5%,57.65%
1,2,55,11.7%,68.52%
2,3,7,1.5%,71.43%


## Check weighted

In [69]:
xgb_reg_weights = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=35,
    subsample=0.65,
    colsample_bytree=0.68,
    reg_alpha=5.28,
    reg_lambda=1.3,
    min_child_weight=5.08,
    gamma=0.0085,
    n_jobs=-1,
    random_state=16,
)

weighted_xgb = TemporalDecaySampleWeightRegressor(
    estimator=xgb_reg_weights,
    dates=df_dev["GAME_DATE"],
    lambda_=SAMPLE_WEIGHT_LAMBDA,
)

cv_results_weights = cross_validate(
    weighted_xgb,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost with sample weights (per-fold decay)")
print_metrics(cv_results_weights)

XGBoost with sample weights (per-fold decay)
Train MSE: 274.76685
Validation MSE: 311.25188

Train RMSE: 16.57601
Validation RMSE: 17.50616

Train MAE: 13.16031
Validation MAE: 13.76440

Train R2: 0.04745
Validation R2: -0.04770

Train OU_Betting_Accuracy: 58.18%
Validation OU_Betting_Accuracy: 52.65%

Train OU_Betting_Accuracy_Edge_2: 69.39%
Validation OU_Betting_Accuracy_Edge_2: 57.70%

Train OU_Betting_Accuracy_Edge_4: 87.80%
Validation OU_Betting_Accuracy_Edge_4: 40.00%



In [70]:
weighted_xgb.fit(X_dev, y_dev)

y_pred_test_error = weighted_xgb.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 311.99416
RMSE: 17.66336
MAE: 13.81307
OU_Betting_Accuracy: 52.06%
OU_Betting_Accuracy_Edge_2: 54.80%
OU_Betting_Accuracy_Edge_4: 61.82%


In [71]:
results_df, y_pred_test_error = evaluate_error_thresholds(
    model=weighted_xgb,
    X_test=X_test_final,
    y_test_error=y_test_final,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,471,100.0%,52.06%
1,1,364,77.3%,51.55%
2,2,255,54.1%,54.80%
3,3,140,29.7%,56.12%
4,4,55,11.7%,61.82%
5,5,15,3.2%,73.33%
6,6,6,1.3%,66.67%
7,7,0,0.0%,nan%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


In [72]:
def fit_and_predict_xgb_weights_day_by_day(train_df, test_df):
    base_model = XGBRegressor(**xgb_reg_weights.get_params())
    model = TemporalDecaySampleWeightRegressor(
        estimator=base_model,
        dates=train_df["GAME_DATE"],
        lambda_=SAMPLE_WEIGHT_LAMBDA,
    )

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_weights, day_by_day_weights_thresholds = run_day_by_day_walk_forward_evaluation(
    label="XGBoost with sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_weights_day_by_day,
    max_games=TRAIN_GAMES,
)


XGBoost with sample weights mean day-by-day OU_Betting_Accuracy: 56.16%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-01-27 00:00:00,5000,7,2021-12-14 00:00:00,2026-01-26 00:00:00,66.67%
1,2026-01-28 00:00:00,5000,9,2021-12-15 00:00:00,2026-01-27 00:00:00,88.89%
2,2026-01-29 00:00:00,5000,8,2021-12-16 00:00:00,2026-01-28 00:00:00,62.50%
3,2026-01-30 00:00:00,5000,9,2021-12-17 00:00:00,2026-01-29 00:00:00,12.50%
4,2026-01-31 00:00:00,5000,6,2021-12-19 00:00:00,2026-01-30 00:00:00,33.33%
5,2026-02-01 00:00:00,5000,10,2021-12-20 00:00:00,2026-01-31 00:00:00,80.00%
6,2026-02-02 00:00:00,5000,4,2021-12-22 00:00:00,2026-02-01 00:00:00,50.00%
7,2026-02-03 00:00:00,5000,10,2021-12-22 00:00:00,2026-02-02 00:00:00,60.00%
8,2026-02-04 00:00:00,5000,7,2021-12-23 00:00:00,2026-02-03 00:00:00,57.14%
9,2026-02-05 00:00:00,5000,8,2021-12-26 00:00:00,2026-02-04 00:00:00,50.00%


XGBoost with sample weights thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,315,66.9%,59.42%
1,2,192,40.8%,60.11%
2,3,101,21.4%,65.00%


# Optuna

In [73]:
study = tune_xgb_error_line_optuna(
    X=X_dev,
    y=y_dev,
    sample_weight_dates=df_dev["GAME_DATE"],
    tune_sample_weight_lambda=True,
    sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    splits=splits,
    n_trials=80,
    timeout=4.5 * 3600,
    # timeout=600,

    objective_name="reg:squarederror",
    study_name="xgb_error_line_mae",
)

best_trial_lexi = select_best_trial_lexicographic(
    study,
    mae_tolerance_abs=0.15,
)

print("Optuna best by MAE only")
print("Trial:", study.best_trial.number)
print("Best CV MAE:", study.best_value)
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", study.best_trial.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", study.best_trial.user_attrs.get("mean_ou_acc_edge_4"))

print("\nSelected trial after MAE-first / OU-second ranking")
print("Trial:", best_trial_lexi.number)
print("CV MAE:", best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value))
print("Mean RMSE:", best_trial_lexi.user_attrs.get("mean_rmse"))
print("Mean R2:", best_trial_lexi.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", best_trial_lexi.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_4"))
print("Median best_iteration:", best_trial_lexi.user_attrs.get("median_best_iteration"))
print("Params:")
for k, v in best_trial_lexi.params.items():
    print(f"{k}: {v}")

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)

candidates_df = summarize_lexicographic_candidates(
    study,
    mae_tolerance_abs=0.15,
)
display(
    candidates_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)


[I 2026-04-05 23:07:27,841] A new study created in memory with name: xgb_error_line_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-04-05 23:16:51,754] Trial 0 finished with value: 13.582660314595188 and parameters: {'max_depth': 2, 'min_child_weight': 18.346704707583235, 'gamma': 1.6970342240854253, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.5123279759064977, 'learning_rate': 0.011926786034588454, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 1.897469395521307, 'sample_weight_lambda': 0.00013824509574177668}. Best is trial 0 with value: 13.582660314595188.
[I 2026-04-05 23:26:13,348] Trial 1 finished with value: 13.628476563080053 and parameters: {'max_depth': 4, 'min_child_weight': 20.290108931287893, 'gamma': 0.3261777842309625, 'subsample': 0.8390562044499359, 'colsample_bytree': 0.4213034780854249, 'learning_rate': 0.012620826760486504, 'reg_alpha': 0.09307011182812809, 'reg_lambda': 15.258811505244246, 'sample_weight_lambda': 0.000848258413826022}. Best is trial 0 with value: 13.582660314595188.
[I 2026-04-05 23:32:36,518] Trial 2 finished with value: 13.637662898368172 and parameters: {'

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,9,13.3474,17.1795,-0.0112,57.72%,51.48%,0.490655,31.98%,62,56,4,15.509831,1.261227,0.669576,0.722762,0.049743,3.380961,1.306366,0.001629
1,14,13.3685,17.0691,0.0033,58.86%,60.08%,0.545072,43.26%,59,42,4,5.583248,1.241355,0.620974,0.609037,0.058679,0.336614,34.472110,0.008982
2,27,13.3913,17.1344,-0.0051,59.39%,54.66%,0.434465,24.80%,69,42,3,12.703873,1.054800,0.641106,0.742931,0.036916,2.119119,12.854323,0.006737
3,25,13.3948,17.1492,-0.0070,56.44%,60.55%,0.615228,36.15%,55,34,3,13.287691,1.119869,0.651614,0.626430,0.048443,2.338918,13.555306,0.007291
4,15,13.3968,17.0854,0.0005,61.16%,49.29%,0.368305,39.38%,76,17,4,5.560352,2.202644,0.623168,0.535979,0.058098,0.371908,48.140084,0.009628
5,33,13.4066,17.1620,-0.0084,58.47%,54.03%,0.427990,37.74%,76,47,3,15.866396,0.514186,0.678038,0.604777,0.051724,1.420588,14.947754,0.007076
6,24,13.4215,17.1867,-0.0113,57.58%,47.12%,0.309072,27.79%,89,11,4,6.305363,2.053492,0.600430,0.475421,0.035537,0.442266,20.904885,0.003964
7,32,13.4244,17.1639,-0.0082,54.28%,46.77%,0.553511,51.62%,57,37,3,9.876180,1.529895,0.654421,0.662856,0.048227,2.032355,14.562141,0.005650
8,16,13.4246,17.0803,0.0016,57.92%,39.58%,0.396474,35.85%,68,27,4,5.708433,1.292869,0.746626,0.623856,0.042234,4.517737,27.084743,0.009988
9,4,13.4264,17.1932,-0.0127,59.49%,29.30%,0.194411,17.69%,103,39,3,27.769564,0.863656,0.554592,0.697083,0.040937,0.028904,9.594362,0.000480


,trial,value_mae,mean_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,mae_cutoff,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,15,13.3968,13.3968,17.0854,0.0005,61.16%,49.29%,0.368305,39.38%,76,17,13.497433,4,5.560352,2.202644,0.623168,0.535979,0.058098,0.371908,48.140084,0.009628
1,4,13.4264,13.4264,17.1932,-0.0127,59.49%,29.30%,0.194411,17.69%,103,39,13.497433,3,27.769564,0.863656,0.554592,0.697083,0.040937,0.028904,9.594362,0.000480
2,27,13.3913,13.3913,17.1344,-0.0051,59.39%,54.66%,0.434465,24.80%,69,42,13.497433,3,12.703873,1.054800,0.641106,0.742931,0.036916,2.119119,12.854323,0.006737
3,14,13.3685,13.3685,17.0691,0.0033,58.86%,60.08%,0.545072,43.26%,59,42,13.497433,4,5.583248,1.241355,0.620974,0.609037,0.058679,0.336614,34.472110,0.008982
4,19,13.4885,13.4885,17.2386,-0.0172,58.74%,38.06%,0.453274,26.94%,56,27,13.497433,4,11.155326,2.077078,0.789054,0.780981,0.023995,4.705250,1.035137,0.006487
5,31,13.4810,13.4810,17.2260,-0.0163,58.51%,53.02%,0.454841,17.96%,56,26,13.497433,3,12.842292,1.078897,0.659986,0.350767,0.047040,3.077776,12.500404,0.007314
6,33,13.4066,13.4066,17.1620,-0.0084,58.47%,54.03%,0.427990,37.74%,76,47,13.497433,3,15.866396,0.514186,0.678038,0.604777,0.051724,1.420588,14.947754,0.007076
7,11,13.4880,13.4880,17.2758,-0.0223,58.35%,57.98%,0.325397,20.00%,94,63,13.497433,3,8.778981,0.773128,0.681748,0.704703,0.032695,0.021864,5.033934,0.001556
8,28,13.4936,13.4936,17.2706,-0.0210,58.33%,52.18%,0.473175,50.00%,71,76,13.497433,3,17.078235,1.457009,0.583702,0.736548,0.038793,2.963621,2.153480,0.001213
9,16,13.4246,13.4246,17.0803,0.0016,57.92%,39.58%,0.396474,35.85%,68,27,13.497433,4,5.708433,1.292869,0.746626,0.623856,0.042234,4.517737,27.084743,0.009988


In [74]:
def fit_and_predict_optuna_day_by_day(train_df, test_df):
    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model = fit_best_xgb_error_line(
        X_dev=X_train,
        y_dev=y_train,
        sample_weight_dates=train_df["GAME_DATE"],
        sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
        trial=best_trial_lexi,
        objective_name="reg:squarederror",
    )
    return model.predict(X_test)

day_by_day_optuna, day_by_day_optuna_thresholds = run_day_by_day_walk_forward_evaluation(
    label="Optuna-selected XGBoost",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_optuna_day_by_day,
    max_games=TRAIN_GAMES,
)

Optuna-selected XGBoost mean day-by-day OU_Betting_Accuracy: 53.07%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-01-27 00:00:00,5000,7,2021-12-14 00:00:00,2026-01-26 00:00:00,66.67%
1,2026-01-28 00:00:00,5000,9,2021-12-15 00:00:00,2026-01-27 00:00:00,55.56%
2,2026-01-29 00:00:00,5000,8,2021-12-16 00:00:00,2026-01-28 00:00:00,62.50%
3,2026-01-30 00:00:00,5000,9,2021-12-17 00:00:00,2026-01-29 00:00:00,25.00%
4,2026-01-31 00:00:00,5000,6,2021-12-19 00:00:00,2026-01-30 00:00:00,33.33%
5,2026-02-01 00:00:00,5000,10,2021-12-20 00:00:00,2026-01-31 00:00:00,80.00%
6,2026-02-02 00:00:00,5000,4,2021-12-22 00:00:00,2026-02-01 00:00:00,75.00%
7,2026-02-03 00:00:00,5000,10,2021-12-22 00:00:00,2026-02-02 00:00:00,40.00%
8,2026-02-04 00:00:00,5000,7,2021-12-23 00:00:00,2026-02-03 00:00:00,57.14%
9,2026-02-05 00:00:00,5000,8,2021-12-26 00:00:00,2026-02-04 00:00:00,50.00%


Optuna-selected XGBoost thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,358,76.0%,54.99%
1,2,239,50.7%,58.12%
2,3,148,31.4%,63.45%


In [75]:
total_df = df_dev.tail(TRAIN_GAMES)

In [76]:
X_dev = total_df.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(total_df[TARGET_COL], errors="coerce")
sample_weight_dates_dev = total_df["GAME_DATE"]


In [77]:
best_model = fit_best_xgb_error_line(
    X_dev=X_dev,
    y_dev=y_dev,
    sample_weight_dates=sample_weight_dates_dev,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

y_pred_test_error = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 315.25324
RMSE: 17.75537
MAE: 13.85853
OU_Betting_Accuracy: 52.93%
OU_Betting_Accuracy_Edge_2: 53.38%
OU_Betting_Accuracy_Edge_4: 53.75%


In [78]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics

df_to_train_split_rows = df_to_train.copy()
df_to_train_split_rows = df_to_train_split_rows.tail(TRAIN_GAMES)

X_full = df_to_train_split_rows.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(df_to_train_split_rows[TARGET_COL], errors="coerce")
sample_weight_dates_full = df_to_train_split_rows["GAME_DATE"]

production_model = fit_best_xgb_error_line(
    X_dev=X_full,
    y_dev=y_full,
    sample_weight_dates=sample_weight_dates_full,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(df_to_train_split_rows["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"five_seasons_xgb_line_error_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="five_seasons_line_error",
        prediction_source="five_seasons_xgb_line_error",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=best_trial_lexi.params,
        selected_trial_number=best_trial_lexi.number,
        mean_best_iteration=best_trial_lexi.user_attrs.get("mean_best_iteration"),
        median_best_iteration=best_trial_lexi.user_attrs.get("median_best_iteration"),
        cv_mae=float(best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value)),
        cv_rmse=best_trial_lexi.user_attrs.get("mean_rmse"),
        cv_ou_acc=best_trial_lexi.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=df_to_train_split_rows["GAME_DATE"].min().to_pydatetime(),
        train_date_max=df_to_train_split_rows["GAME_DATE"].max().to_pydatetime(),
        train_games= TRAIN_GAMES,
        sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/5_seasons/",
    metadata=metadata,
)

print(
    f"Production model trained on {len(X_full)} rows using fixed n_estimators from median_best_iteration."
)
print("Saved model :", model_path)
print("Saved metadata:", meta_path)

Production model trained on 5000 rows using fixed n_estimators from median_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/5_seasons/five_seasons_xgb_line_error_04_04_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/5_seasons/five_seasons_xgb_line_error_04_04_26.meta.json


In [79]:
best_trial_lexi.user_attrs.get("median_best_iteration")


17